# US County Federal Information Processing Series (FIPS) Code Lookup ETL Pipeline

American National Standards Institute (ANSI), Federal Information Processing Series (FIPS), and Other Standardized Geographic Codes<br>
https://www.census.gov/library/reference/code-lists/ansi.html?utm_source=copilot.com#cou

The following code downloads pipe dilimited (|) text files containing US fips code to county name mapping.

In [ ]:
import os
import requests
from bs4 import BeautifulSoup

# print(os.getcwd())

In [ ]:
BASE_URL = "https://www2.census.gov/geo/docs/reference/codes2020/cou/"

save_dir = "data/county_fips_2020"

# Create a folder to store the files
os.makedirs(save_dir, exist_ok=True)

# Get directory listing
resp = requests.get(BASE_URL)
soup = BeautifulSoup(resp.text, "html.parser")

links = []

for a in soup.find_all("a"):
    href = a.get("href")   # <-- safe access, avoids KeyError
    if href and href.endswith(".txt"):
        links.append(href)

for filename in links:
    file_url = BASE_URL + filename
    print("Downloading:", filename)
    file_resp = requests.get(file_url)

    with open(os.path.join(save_dir, filename), "wb") as f:
        f.write(file_resp.content)

Transform and Export Data to Postgresql

In [ ]:
# !pip install sqlalchemy
# !pip install psycopg2

In [ ]:
import pandas as pd

# FUNCTION: Read & Filter a CSV in chunks
def load_filtered_chunks(
    file_path,
    chunksize=100_000
):
    """
    Reads a pipe delimited TXT file in chunks, 
    making sure specific columns retain leading
    0s as needed.
    """
    string_cols = {"STATEFP": str, "COUNTYFP": str}
    for chunk in pd.read_csv(
        file_path,
        dtype = string_cols,
        chunksize=chunksize,
        low_memory=False,
        sep="|"
    ):
        # Create county code lookup
        concat_cols = ["STATEFP", "COUNTYFP"]
        chunk['AREA_FIPS'] = chunk[concat_cols].astype(str).fillna("").agg("".join, axis=1).str.lstrip("0")
        
        # Normalize column names to lowercase to match postgresql
        chunk.columns = [col.lower() for col in chunk.columns]

        yield chunk

In [ ]:
# from sqlalchemy import create_engine

# FUNCTION: Upload a chunk to PostgreSQL
def upload_chunk(df, table_name, engine):
    df.to_sql(
        table_name,
        engine,
        if_exists="append",
        index=False,
        method="multi",
        chunksize=10_000
    )

In [ ]:
# FUNCTION: Process a file
def process_file(file_path, table_name, engine):
    for chunk in load_filtered_chunks(file_path):
        upload_chunk(chunk, table_name, engine)

In [ ]:
# ESTABLISH THE POSTGRESQL CONNECTION

import glob
from sqlalchemy import create_engine

# PostgreSQL connection parameters
from utility import portnumber, host, database, username, password

# Connection parameters
HOST = host
PORT = portnumber
DBNAME = database
USER = username
PASSWORD = password

# Ensure Server is running and Destination Table exists within the database
engine = create_engine(f"postgresql+psycopg2://{USER}:{PASSWORD}@{HOST}:{PORT}/{DBNAME}")


In [ ]:
# PROCESS EACH FILE IN THE DIRECTORY
files = glob.glob("data/county_fips_2020/*.txt") 

for file in files:
    process_file(
        file_path=file,
        table_name="fips_code_lookup",
        engine=engine
    )